# Day 6 — Error Handling: Retries, Timeouts, and Failure Flows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/kestra-certified/notebooks/day-06-error-handling.ipynb#scrollTo=a1c2e3f4)

**Course:** Kestra for Data Engineers  
**Badge:** Review  

Production pipelines fail. Networks hiccup, APIs rate-limit, schemas drift. Today you'll build all three layers of Kestra's error-handling model: task-level retries, flow-level error handlers, and `allowFailure` for non-critical steps.

**By the end of this notebook you will:**
- Configure exponential backoff retries on individual tasks
- Set task-level timeouts and understand the TIMEOUT execution state
- Write flow-level `errors:` blocks that fire on any failure
- Use `allowFailure: true` to keep a flow running past non-critical failures
- Understand Kestra's execution state machine: RUNNING → SUCCESS / FAILED / TIMEOUT / KILLED

In [ ]:
%pip install -q pyyaml

## 1. Kestra's Execution State Machine

Every Kestra execution moves through a defined set of states:

```
CREATED → RUNNING ─┬→ SUCCESS
                   ├→ FAILED   (task threw an exception)
                   ├→ TIMEOUT  (task exceeded timeout duration)
                   ├→ KILLED   (manually stopped)
                   └→ WARNING  (allowFailure task failed but flow continued)
```

Error handling in Kestra operates at two levels:
- **Task level:** retries + timeout per task
- **Flow level:** `errors:` block that runs when any task fails

In [ ]:
import yaml

# Visualise execution state transitions as a dict
states = {
    "CREATED": "Execution queued, not yet started",
    "RUNNING": "At least one task is executing",
    "SUCCESS": "All tasks completed without error",
    "FAILED": "A task failed (and no retry succeeded)",
    "TIMEOUT": "A task exceeded its timeout duration",
    "KILLED": "Manually stopped via UI or API",
    "WARNING": "An allowFailure task failed; flow continued"
}

print(f"{'State':<12} Description")
print("-" * 60)
for state, desc in states.items():
    print(f"{state:<12} {desc}")

## 2. Task-Level Retries — Exponential Backoff

Add a `retry:` block to any task to automatically re-execute it on failure:

```yaml
retry:
  type: exponential    # or: constant
  maxAttempts: 3       # total attempts (including the first)
  delay: PT5S          # initial delay (ISO 8601 duration)
  multiplier: 2.0      # each subsequent delay is multiplied by this
  maxDelay: PT60S      # cap on individual delay
```

Attempt schedule for `delay: PT5S, multiplier: 2.0, maxAttempts: 3`:
- Attempt 1: immediate
- Attempt 2: wait 5s
- Attempt 3: wait 10s
- → FAILED if attempt 3 also fails

In [ ]:
retry_flow = {
    "id": "retry-demo",
    "namespace": "tutorial.day06",
    "description": "Demonstrates exponential backoff on a flaky HTTP task",
    "tasks": [
        {
            "id": "fetch_with_retry",
            "type": "io.kestra.plugin.core.http.Request",
            "uri": "https://jsonplaceholder.typicode.com/posts/1",
            "method": "GET",
            "retry": {
                "type": "exponential",
                "maxAttempts": 4,
                "delay": "PT5S",
                "multiplier": 2.0,
                "maxDelay": "PT30S"
            }
        },
        {
            "id": "log_success",
            "type": "io.kestra.plugin.core.log.Log",
            "message": "Fetch succeeded after retries — HTTP {{ outputs.fetch_with_retry.code }}"
        }
    ]
}

print(yaml.dump(retry_flow, default_flow_style=False, sort_keys=False))

# Simulate the delay schedule
delay = 5.0
multiplier = 2.0
max_delay = 30.0
max_attempts = 4

print("\nAttempt schedule:")
total = 0.0
for attempt in range(1, max_attempts + 1):
    if attempt == 1:
        print(f"  Attempt {attempt}: immediate")
    else:
        wait = min(delay * (multiplier ** (attempt - 2)), max_delay)
        total += wait
        print(f"  Attempt {attempt}: wait {wait:.0f}s (cumulative: {total:.0f}s)")

> **Constant retry** (`type: constant`) waits the same `delay` between every attempt. Use exponential backoff for rate-limited APIs and constant retry for deterministic operations like database connections.

## 3. Task-Level Timeouts

A `timeout:` property sets the maximum duration a single task attempt may run. If exceeded, the task transitions to TIMEOUT state and the retry logic applies (if configured).

Duration format: ISO 8601 — `PT30S` = 30 seconds, `PT5M` = 5 minutes, `PT2H` = 2 hours

In [ ]:
timeout_flow = {
    "id": "timeout-demo",
    "namespace": "tutorial.day06",
    "tasks": [
        {
            "id": "slow_query",
            "type": "io.kestra.plugin.scripts.python.Commands",
            "timeout": "PT30S",
            "retry": {
                "type": "constant",
                "maxAttempts": 2,
                "delay": "PT10S"
            },
            "beforeCommands": [],
            "script": (
                "import time\n"
                "# This would timeout in Kestra if it ran > 30s\n"
                "print('Starting slow operation...')\n"
                "time.sleep(5)  # reduced for local simulation\n"
                "print('Done')\n"
            )
        },
        {
            "id": "fast_task",
            "type": "io.kestra.plugin.core.log.Log",
            "timeout": "PT5S",
            "message": "This log task will timeout if it takes > 5 seconds"
        }
    ]
}

print(yaml.dump(timeout_flow, default_flow_style=False, sort_keys=False))

# ISO 8601 duration parser (Python 3.11+ has datetime.timedelta.fromisoformat)
import re

def parse_iso8601_duration(s):
    """Parse PT30S, PT5M, PT2H into seconds."""
    m = re.match(r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?', s)
    h, m_, s_ = (int(x) if x else 0 for x in m.groups())
    return h * 3600 + m_ * 60 + s_

examples = ['PT30S', 'PT5M', 'PT2H', 'PT1H30M', 'PT10S']
print("\nISO 8601 durations:")
for d in examples:
    print(f"  {d:<10} = {parse_iso8601_duration(d)}s")

## 4. Flow-Level Error Handlers — The errors: Block

The `errors:` key at the flow level defines tasks that run **only when the flow fails**. Use them for:
- Sending failure alerts (email, Teams, PagerDuty)
- Cleaning up partial state
- Rolling back a database transaction
- Logging enriched failure context

Error tasks receive the same Pebble context including `{{ execution.id }}`, `{{ flow.id }}`, and `{{ taskrun.attemptsCount }}`.

In [ ]:
error_handler_flow = {
    "id": "error-handler-demo",
    "namespace": "tutorial.day06",
    "description": "Demonstrates flow-level error handler with alert notification",
    "tasks": [
        {
            "id": "risky_task",
            "type": "io.kestra.plugin.scripts.python.Commands",
            "beforeCommands": [],
            "script": (
                "import random\n"
                "# Simulate a flaky task\n"
                "if random.random() < 0.5:\n"
                "    raise ValueError('Simulated data quality failure')\n"
                "print('Task succeeded')\n"
            )
        },
        {
            "id": "downstream_task",
            "type": "io.kestra.plugin.core.log.Log",
            "message": "Processing complete"
        }
    ],
    "errors": [
        {
            "id": "alert_on_failure",
            "type": "io.kestra.plugin.core.http.Request",
            "uri": "https://httpbin.org/post",
            "method": "POST",
            "headers": {"Content-Type": "application/json"},
            "body": (
                "{\"text\": \"Flow {{ flow.id }} FAILED on execution {{ execution.id }} "
                "at {{ execution.startDate }}\", "
                "\"severity\": \"HIGH\"}"
            )
        },
        {
            "id": "log_failure",
            "type": "io.kestra.plugin.core.log.Log",
            "level": "ERROR",
            "message": "FAILURE: {{ flow.id }} | execution: {{ execution.id }} | triggered at {{ execution.startDate }}"
        }
    ]
}

print(yaml.dump(error_handler_flow, default_flow_style=False, sort_keys=False))

> Error tasks in `errors:` run sequentially. If an error task itself fails, Kestra logs it but does not recurse into a nested error handler — the original failure state is preserved.

## 5. allowFailure — Non-Critical Steps

`allowFailure: true` on a task tells Kestra: "if this task fails, mark it as WARNING and continue the flow." The execution ends as WARNING (not FAILED) if any `allowFailure` task failed but all required tasks succeeded.

Use cases:
- Optional data enrichment (best-effort, not blocking)
- Notification tasks that shouldn't block a pipeline
- Schema-drift checks where a mismatch should warn, not halt

In [ ]:
allow_failure_flow = {
    "id": "allow-failure-demo",
    "namespace": "tutorial.day06",
    "description": "Optional enrichment step that does not block the critical path",
    "tasks": [
        {
            "id": "critical_ingest",
            "type": "io.kestra.plugin.core.http.Request",
            "uri": "https://jsonplaceholder.typicode.com/posts/1",
            "method": "GET"
        },
        {
            "id": "optional_geo_enrich",
            "type": "io.kestra.plugin.core.http.Request",
            "allowFailure": True,
            "uri": "https://this-endpoint-does-not-exist.example.com/enrich",
            "method": "POST",
            "retry": {
                "type": "constant",
                "maxAttempts": 2,
                "delay": "PT3S"
            }
        },
        {
            "id": "critical_write",
            "type": "io.kestra.plugin.core.log.Log",
            "message": (
                "Ingest complete (enrichment status: "
                "{{ outputs.optional_geo_enrich.code | default('N/A') }})"
            )
        }
    ]
}

print(yaml.dump(allow_failure_flow, default_flow_style=False, sort_keys=False))

> When `optional_geo_enrich` fails, `critical_write` still runs because it doesn't depend on that output being successful. The execution finishes with state **WARNING** — visible in the Kestra UI as a yellow indicator rather than red.

## 6. Simulate Error Handling Logic Locally

In [ ]:
import random, time

def simulate_task_with_retry(task_name, max_attempts, delay_s, multiplier, fail_prob=0.7):
    """Simulate a flaky task with exponential backoff retry."""
    print(f"\n[{task_name}] Starting (fail_prob={fail_prob})")
    current_delay = delay_s
    for attempt in range(1, max_attempts + 1):
        if random.random() > fail_prob:
            print(f"  Attempt {attempt}: SUCCESS")
            return True
        else:
            if attempt < max_attempts:
                print(f"  Attempt {attempt}: FAILED — retrying in {current_delay:.1f}s")
                # time.sleep(current_delay)  # skipped for notebook speed
                current_delay = min(current_delay * multiplier, 30)
            else:
                print(f"  Attempt {attempt}: FAILED — no more retries")
    return False

random.seed(42)

# Critical task with retry
result = simulate_task_with_retry("fetch_data", max_attempts=4, delay_s=5, multiplier=2.0, fail_prob=0.6)

if result:
    print("\nFlow state: SUCCESS")
    # Optional task with allowFailure
    optional_ok = simulate_task_with_retry("optional_enrich", max_attempts=2, delay_s=3, multiplier=1.0, fail_prob=0.9)
    if not optional_ok:
        print("  optional_enrich failed — allowFailure=True, continuing")
    print("  critical_write: executed")
    flow_state = "WARNING" if not optional_ok else "SUCCESS"
    print(f"\nFinal flow state: {flow_state}")
else:
    print("\nFlow state: FAILED → triggering errors: block")
    print("  alert_on_failure: POST to notification endpoint")
    print("  log_failure: ERROR level log written")

## 7. Complete Error-Resilient Flow

In [ ]:
# Production-grade error handling pattern
resilient_flow = {
    "id": "resilient-pipeline",
    "namespace": "tutorial.day06",
    "description": "All three error handling layers: retry + timeout + errors block + allowFailure",
    "tasks": [
        {
            "id": "ingest",
            "type": "io.kestra.plugin.core.http.Request",
            "uri": "https://jsonplaceholder.typicode.com/posts/1",
            "timeout": "PT30S",
            "retry": {
                "type": "exponential",
                "maxAttempts": 3,
                "delay": "PT5S",
                "multiplier": 2.0
            }
        },
        {
            "id": "transform",
            "type": "io.kestra.plugin.scripts.python.Commands",
            "timeout": "PT2M",
            "beforeCommands": [],
            "env": {"BODY": "{{ outputs.ingest.body }}"},
            "script": (
                "import json, os\n"
                "post = json.loads(os.environ['BODY'])\n"
                "result = {'id': post['id'], 'title_len': len(post['title']), 'has_body': bool(post['body'])}\n"
                "Kestra.outputs(result)\n"
            )
        },
        {
            "id": "optional_audit_log",
            "type": "io.kestra.plugin.core.http.Request",
            "allowFailure": True,
            "uri": "https://httpbin.org/post",
            "method": "POST",
            "headers": {"Content-Type": "application/json"},
            "body": "{\"event\": \"pipeline.run\", \"execution\": \"{{ execution.id }}\"}"
        },
        {
            "id": "log_result",
            "type": "io.kestra.plugin.core.log.Log",
            "message": (
                "post_id={{ outputs.transform.vars.id }} | "
                "title_len={{ outputs.transform.vars.title_len }} | "
                "has_body={{ outputs.transform.vars.has_body }}"
            )
        }
    ],
    "errors": [
        {
            "id": "notify_failure",
            "type": "io.kestra.plugin.core.log.Log",
            "level": "ERROR",
            "message": (
                "ALERT: {{ flow.id }} failed on execution {{ execution.id }}. "
                "Check Kestra UI for task-level error details."
            )
        }
    ]
}

print(yaml.dump(resilient_flow, default_flow_style=False, sort_keys=False))

## 8. Error Handling Decision Matrix

In [ ]:
# Decision guide: which error handling tool to use when
decision_matrix = [
    {
        "scenario": "Flaky external API",
        "tool": "retry (exponential)",
        "config": "maxAttempts: 4, delay: PT5S, multiplier: 2.0"
    },
    {
        "scenario": "Long-running ML job",
        "tool": "timeout",
        "config": "timeout: PT2H"
    },
    {
        "scenario": "Send Slack on failure",
        "tool": "errors: block",
        "config": "errors: [{id: alert, type: http.Request, ...}]"
    },
    {
        "scenario": "Optional enrichment",
        "tool": "allowFailure: true",
        "config": "allowFailure: true"
    },
    {
        "scenario": "DB connection on cold start",
        "tool": "retry (constant)",
        "config": "maxAttempts: 5, delay: PT3S, type: constant"
    }
]

print(f"{'Scenario':<30} {'Tool':<20} Config")
print("-" * 90)
for row in decision_matrix:
    print(f"{row['scenario']:<30} {row['tool']:<20} {row['config']}")

## Challenge

Build a Kestra flow with all three error handling layers:

1. A `fetch` task that hits `https://jsonplaceholder.typicode.com/posts/1` with `maxAttempts: 3`, `delay: PT2S`, `multiplier: 2.0`, and `timeout: PT15S`
2. A Python `validate` task that checks `{{ outputs.fetch.code }} == 200` and raises `ValueError` if not — also with `timeout: PT10S`
3. An `optional_cache_write` task with `allowFailure: true` that logs `"Cache written"`
4. A `log_success` task that logs the HTTP body length
5. An `errors:` block with a task that logs `"PIPELINE FAILED: {{ flow.id }} / {{ execution.id }}"`

In [ ]:
# Your solution here
challenge_flow = {
    "id": "resilient-challenge",
    "namespace": "tutorial.day06.challenge",
    "tasks": [
        # Task 1: fetch with retry + timeout
        # Task 2: validate with timeout
        # Task 3: optional_cache_write with allowFailure
        # Task 4: log_success
    ],
    "errors": [
        # Failure alert task
    ]
}
print(yaml.dump(challenge_flow, default_flow_style=False, sort_keys=False))

## Recap

| Mechanism | YAML key | Scope | Use when |
|-----------|----------|-------|----------|
| Retry (exponential) | `retry.type: exponential` | Task | Flaky APIs, transient network errors |
| Retry (constant) | `retry.type: constant` | Task | DB connections, cold starts |
| Timeout | `timeout: PTxS` | Task | Long-running jobs that should be bounded |
| Error handler | `errors: [...]` | Flow | Alerts, cleanup, rollback on any failure |
| Allow failure | `allowFailure: true` | Task | Optional steps that must not block critical path |

**Tip:** Use exponential backoff (`multiplier: 2.0`) for network-dependent tasks — a 5-second base delay with 3 retries gives you 5s, 10s, 20s before failing. This handles transient API rate limits gracefully.

**Tomorrow — Day 7:** Subflows and Namespaces — modular workflow design, reusable child flows, and namespace file inheritance.